# SituatiONION v3: Layer-Wise Causal Tests of Situation Representations

This notebook tests whether GPT-2 XL middle layers transiently encode a situation representation: stable across wording changes but sensitive to changes in agents, recipients, polarity, cause, and time. It replaces the exploratory claims in v2 with controlled measurements and a causal-patching hook.

## Preregistered primary metric

For a base text $x$, a meaning-preserving paraphrase $x^+$, and a meaning-changing counterfactual $x^-$, let $h^{(l)}(x)$ be the pooled residual-stream representation at block $l$. The project-defined situation score is:

$$S_{sit}(l) = E[cos(h^{(l)}(x), h^{(l)}(x^+)) - cos(h^{(l)}(x), h^{(l)}(x^-))].$$

The hypothesis is a reliable local maximum in GPT-2 XL layers 23-26. Geometry and probes locate readable information; activation patching tests whether the state is causally used.

In [2]:
pip install torch transformers scikit-learn pandas matplotlib seaborn


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 2.7 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1
Note: you may need to restart the kernel to use updated packages.


In [4]:
# %pip install torch transformers scikit-learn pandas matplotlib seaborn
from dataclasses import dataclass
from typing import Dict, Sequence

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from transformers import GPT2LMHeadModel, GPT2Tokenizer

sns.set_theme(style='whitegrid')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32
SEED = 7
np.random.seed(SEED); torch.manual_seed(SEED)
print(f'Using {DEVICE} with {DTYPE}.')

Using cpu with torch.float32.


In [2]:
MODEL_NAME = 'gpt2-xl'
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(
    MODEL_NAME, torch_dtype=DTYPE, output_hidden_states=True, output_attentions=True
).to(DEVICE).eval()

N_LAYERS = model.config.n_layer
MID = range(17, 32)
CORE_MID = range(23, 27)
print(f'{MODEL_NAME}: {N_LAYERS} blocks; core-middle: {list(CORE_MID)}')

NameError: name 'GPT2Tokenizer' is not defined

## Controlled situation triples

The seed examples are pipeline checks, not a publishable dataset. Expand them to hundreds of balanced triples and split by lexical template and entity names. Every triple has a human-readable situation graph: event roles plus a causal or temporal relation.

In [ ]:
@dataclass(frozen=True)
class SituationTriple:
    identifier: str
    base: str
    paraphrase: str
    counterfactual: str
    agent: str
    recipient: str
    event: str
    object_name: str
    relation: str
    changed_field: str

SITUATIONS = [
    SituationTriple('transfer_01', 'Leo was locked out, so Maya gave him the key.', 'Because Leo could not enter, Maya handed the key to him.', 'Maya was locked out, so Leo gave her the key.', 'Maya', 'Leo', 'give', 'key', 'cause(locked_out(Leo), give(Maya, Leo, key))', 'agent_recipient_cause'),
    SituationTriple('transfer_02', 'Nora lent Omar the map because he was lost.', 'Omar was lost, and Nora handed him a map.', 'Nora lent Omar the map because she was lost.', 'Nora', 'Omar', 'give', 'map', 'cause(lost(Omar), give(Nora, Omar, map))', 'cause'),
    SituationTriple('transfer_03', 'Iris passed Jules the torch after the lights failed.', 'After the lights went out, Iris handed the torch to Jules.', 'After the lights went out, Jules handed the torch to Iris.', 'Iris', 'Jules', 'give', 'torch', 'after(lights_failed, give(Iris, Jules, torch))', 'agent_recipient'),
    SituationTriple('repair_01', 'Sam repaired the bike because it was broken.', 'Because the bike was broken, Sam fixed it.', 'Sam did not repair the bike even though it was broken.', 'Sam', 'bike', 'repair', 'bike', 'cause(broken(bike), repair(Sam, bike))', 'polarity'),
    SituationTriple('repair_02', 'Ava cleaned the spill after the glass fell.', 'After the glass fell, Ava wiped up the spill.', 'Ava cleaned the spill before the glass fell.', 'Ava', 'spill', 'clean', 'spill', 'after(glass_fell, clean(Ava, spill))', 'temporal_relation'),
    SituationTriple('rescue_01', 'Kai rescued the dog because it was trapped.', 'The dog was trapped, so Kai saved it.', 'The dog rescued Kai because he was trapped.', 'Kai', 'dog', 'rescue', 'dog', 'cause(trapped(dog), rescue(Kai, dog))', 'agent_recipient'),
    SituationTriple('message_01', 'Mina warned Theo because the bridge was unsafe.', 'Because the bridge was unsafe, Mina alerted Theo.', 'Theo warned Mina because the bridge was unsafe.', 'Mina', 'Theo', 'warn', 'bridge', 'cause(unsafe(bridge), warn(Mina, Theo))', 'agent_recipient'),
    SituationTriple('food_01', 'Ravi gave Uma soup because she was ill.', 'Uma was ill, so Ravi brought her soup.', 'Ravi gave Uma soup because he was ill.', 'Ravi', 'Uma', 'give', 'soup', 'cause(ill(Uma), give(Ravi, Uma, soup))', 'cause'),
]
pd.DataFrame([s.__dict__ for s in SITUATIONS])

In [ ]:
def run_text(text: str) -> Dict[str, object]:
    inputs = tokenizer(text, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True, output_attentions=True, use_cache=False)
    return {
        'text': text,
        'input_ids': inputs.input_ids.detach().cpu(),
        # Index 0 is embeddings. Index l + 1 is output after Transformer block l.
        'hidden_states': tuple(x.detach().float().cpu() for x in outputs.hidden_states),
        'attentions': tuple(x.detach().float().cpu() for x in outputs.attentions),
        'logits': outputs.logits.detach().float().cpu(),
    }

def layer_vector(run: Dict[str, object], block_layer: int) -> np.ndarray:
    # Mean pooling avoids the final-token-only confound in differently worded inputs.
    return run['hidden_states'][block_layer + 1][0].mean(dim=0).numpy()

def cosine(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

def cache_runs(situations: Sequence[SituationTriple]):
    return {s.identifier: {
        'base': run_text(s.base),
        'paraphrase': run_text(s.paraphrase),
        'counterfactual': run_text(s.counterfactual),
    } for s in situations}

RUNS = cache_runs(SITUATIONS)

In [ ]:
def situation_curve(situations, runs) -> pd.DataFrame:
    rows = []
    for layer in range(N_LAYERS):
        paraphrase, counterfactual = [], []
        for situation in situations:
            item = runs[situation.identifier]
            base = layer_vector(item['base'], layer)
            paraphrase.append(cosine(base, layer_vector(item['paraphrase'], layer)))
            counterfactual.append(cosine(base, layer_vector(item['counterfactual'], layer)))
        difference = np.subtract(paraphrase, counterfactual)
        rows.append({
            'layer': layer,
            'paraphrase_similarity': np.mean(paraphrase),
            'counterfactual_similarity': np.mean(counterfactual),
            'situation_score': np.mean(difference),
            'situation_score_se': np.std(difference, ddof=1) / np.sqrt(len(difference)),
        })
    return pd.DataFrame(rows)

scores = situation_curve(SITUATIONS, RUNS)
display(scores.loc[[scores.situation_score.idxmax()]])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(scores.layer, scores.paraphrase_similarity, label='Same situation: paraphrase')
ax.plot(scores.layer, scores.counterfactual_similarity, label='Changed situation: counterfactual')
ax.plot(scores.layer, scores.situation_score, linewidth=3, label='Situation score')
ax.axvspan(min(CORE_MID), max(CORE_MID), color='orange', alpha=0.16, label='Core-middle hypothesis')
ax.set(xlabel='GPT-2 XL block', ylabel='Cosine similarity / score', title='Layer-wise situation geometry')
ax.legend(); plt.show()

## Linear probes and attention-span diagnostic

Tenney et al. (2019) locate readable information with a learned scalar mixture, $h = gamma sum_l s_l h_l$ and $s_l = softmax(a)_l$. This notebook instead fits separate linear probes per layer, which makes the depth profile visible. The MSPGT preprint defines attention span as:

$$d_{attn}^{(l,h)} = \frac{1}{T}\sum_{t=1}^{T}\sum_{i=1}^{n} A_{t,i}^{(l,h)}|t-i|.$$

Attention span is descriptive supporting evidence. A probe or attention pattern cannot establish causal use.

In [ ]:
def attention_span(attention: torch.Tensor) -> float:
    # attention: (batch, heads, query_positions, key_positions)
    weights = attention[0]
    positions = torch.arange(weights.shape[-1], dtype=weights.dtype)
    distances = (positions[None, :] - positions[:, None]).abs()
    return float((weights * distances.unsqueeze(0)).sum(dim=-1).mean())

attention_scores = pd.DataFrame([{
    'layer': layer,
    'attention_span': np.mean([attention_span(RUNS[s.identifier]['base']['attentions'][layer]) for s in SITUATIONS]),
} for layer in range(N_LAYERS)])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(attention_scores.layer, attention_scores.attention_span)
ax.axvspan(min(CORE_MID), max(CORE_MID), color='orange', alpha=0.16)
ax.set(xlabel='GPT-2 XL block', ylabel='Mean attended distance', title='Attention-span diagnostic')
plt.show()

def probe_curve(situations, runs, label_name: str) -> pd.DataFrame:
    labels = np.array([getattr(s, label_name) for s in situations])
    y = LabelEncoder().fit_transform(labels)
    min_count = min(np.bincount(y))
    if min_count < 2:
        raise ValueError('Add balanced examples: each probe class needs at least two items.')
    cv = StratifiedKFold(n_splits=min(3, min_count), shuffle=True, random_state=SEED)
    rows = []
    for layer in range(N_LAYERS):
        X = np.stack([layer_vector(runs[s.identifier]['base'], layer) for s in situations])
        accuracy = cross_val_score(LogisticRegression(max_iter=4000, class_weight='balanced'), X, y, cv=cv).mean()
        rows.append({'layer': layer, 'accuracy': accuracy})
    return pd.DataFrame(rows)

# Example after expanding SITUATIONS into a balanced dataset:
# event_probe = probe_curve(SITUATIONS, RUNS, 'event')

## Causal test: final-position activation patching

The patch replaces the target run's final-position residual state after a chosen block with the donor state at the same position. For a candidate token $y$, measure:

$$Delta_y = log P(y | patch(h_{donor}^{(l)})) - log P(y | h_{target}^{(l)}).$$

Use matched-length prompts at first. Compare core-middle patches against early, late, random, and lexical-match controls.

In [ ]:
def next_token_logprob(logits: torch.Tensor, token_text: str) -> float:
    token_ids = tokenizer.encode(token_text, add_special_tokens=False)
    if len(token_ids) != 1:
        raise ValueError(f'Use a single GPT-2 token. {token_text!r} becomes {token_ids}.')
    return float(torch.log_softmax(logits[0, -1].float(), dim=-1)[token_ids[0]].cpu())

def patch_final_position(donor_text: str, target_text: str, block_layer: int, candidate_token: str) -> Dict[str, float]:
    donor_inputs = tokenizer(donor_text, return_tensors='pt').to(DEVICE)
    target_inputs = tokenizer(target_text, return_tensors='pt').to(DEVICE)
    if donor_inputs.input_ids.shape[1] != target_inputs.input_ids.shape[1]:
        raise ValueError('Use matched token lengths for this first causal test.')
    with torch.no_grad():
        donor = model(**donor_inputs, use_cache=False)
        baseline = model(**target_inputs, use_cache=False)
    donor_state = donor.hidden_states[block_layer + 1][:, -1:, :].detach()

    def replace_final_position(module, inputs, output):
        hidden = output[0].clone()
        hidden[:, -1:, :] = donor_state
        return (hidden,) + output[1:]

    handle = model.transformer.h[block_layer].register_forward_hook(replace_final_position)
    try:
        with torch.no_grad():
            patched = model(**target_inputs, use_cache=False)
    finally:
        handle.remove()

    baseline_lp = next_token_logprob(baseline.logits, candidate_token)
    patched_lp = next_token_logprob(patched.logits, candidate_token)
    return {'layer': block_layer, 'baseline_logprob': baseline_lp, 'patched_logprob': patched_lp, 'delta_logprob': patched_lp - baseline_lp}

# Example after selecting matched-token-length prompts and a one-token continuation:
# patch_final_position('Leo has the key and can', 'Leo has no key and can', 24, ' enter')

## Decision rule

Claim a core-middle situation regime only if held-out lexical templates show all four results: (1) a reliable local maximum of $S_{sit}(l)$ in the preregistered band; (2) stronger event-role/discourse probes there than relevant controls; (3) compatible attention-span or relation-head changes; and (4) selective, replicated changes in event-consistent next-token probabilities under patching.

This follows the layer-localization evidence in Jawahar et al. (2019), Tenney et al. (2019), Vig (2019), and Clark et al. (2019), while adopting the causal-interpretability warning emphasized by Rogers et al. (2020) and Ferrando et al. (2024). The discourse target is the microstructure/macrostructure distinction of Kintsch and van Dijk (1978, 1983): entities, events, roles, relations, and global topic rather than word overlap alone.